In [1]:
import pandas as pd
import numpy as np

In [2]:
bureau = pd.read_csv("../data/bureau.csv")

print(bureau.shape)
bureau.head()

(1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [3]:
print(bureau.info())
print("\nUnique applicants:", bureau["SK_ID_CURR"].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           str    
 3   CREDIT_CURRENCY         str    
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             str    
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), str(3)
memory usage: 222.6 MB
None

Unique applicants: 305811


In [4]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_ACTIVE_COUNT=("CREDIT_ACTIVE", lambda x: (x == "Active").sum()),
    BUREAU_CLOSED_COUNT=("CREDIT_ACTIVE", lambda x: (x == "Closed").sum()),
    BUREAU_CREDIT_SUM=("AMT_CREDIT_SUM", "sum"),
    BUREAU_CREDIT_SUM_MEAN=("AMT_CREDIT_SUM", "mean"),
    BUREAU_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_OVERDUE_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_OVERDUE_MEAN=("AMT_CREDIT_SUM_OVERDUE", "mean"),
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean")
).reset_index()

In [5]:
print(bureau_agg.shape)
bureau_agg.head()

(305811, 10)


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_CLOSED_COUNT,BUREAU_CREDIT_SUM,BUREAU_CREDIT_SUM_MEAN,BUREAU_DEBT_SUM,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MEAN,BUREAU_DAYS_CREDIT_MEAN
0,100001,7,3,4,1453365.000,207623.571429,596686.5,0.0,0.0,-735.000000
1,100002,8,2,6,865055.565,108131.945625,245781.0,0.0,0.0,-874.000000
2,100003,4,1,3,1017400.500,254350.125000,0.0,0.0,0.0,-1400.750000
3,100004,2,0,2,189037.800,94518.900000,0.0,0.0,0.0,-867.000000
4,100005,3,2,1,657126.000,219042.000000,568408.5,0.0,0.0,-190.666667


In [6]:
application = pd.read_csv("../data/application_train.csv")

enhanced_df = application.merge(
    bureau_agg,
    on="SK_ID_CURR",
    how="left"
)

print("Original:", application.shape)
print("Enhanced:", enhanced_df.shape)

Original: (307511, 122)
Enhanced: (307511, 131)


## Bureau Feature Engineering

The `bureau.csv` dataset contains multiple historical credit records for individual applicants. Since the main application dataset contains one row per applicant, the bureau records were aggregated using `SK_ID_CURR`.

Nine applicant-level features were created, including historical credit count, active and closed credit counts, total credit, outstanding debt, overdue amounts, and average credit history duration.

The aggregated bureau features were merged with the main application dataset using `SK_ID_CURR`. The number of applicants remained unchanged at 307,511, while the number of features increased from 122 to 131.

In [7]:
bureau_features = [
    "BUREAU_LOAN_COUNT",
    "BUREAU_ACTIVE_COUNT",
    "BUREAU_CLOSED_COUNT",
    "BUREAU_CREDIT_SUM",
    "BUREAU_CREDIT_SUM_MEAN",
    "BUREAU_DEBT_SUM",
    "BUREAU_OVERDUE_SUM",
    "BUREAU_OVERDUE_MEAN",
    "BUREAU_DAYS_CREDIT_MEAN"
]

enhanced_df[bureau_features].isnull().sum()

BUREAU_LOAN_COUNT          44020
BUREAU_ACTIVE_COUNT        44020
BUREAU_CLOSED_COUNT        44020
BUREAU_CREDIT_SUM          44020
BUREAU_CREDIT_SUM_MEAN     44021
BUREAU_DEBT_SUM            44020
BUREAU_OVERDUE_SUM         44020
BUREAU_OVERDUE_MEAN        44020
BUREAU_DAYS_CREDIT_MEAN    44020
dtype: int64

In [8]:
enhanced_df[bureau_features] = enhanced_df[bureau_features].fillna(0)

In [9]:
enhanced_df[bureau_features].isnull().sum().sum()

np.int64(0)

## Handling Missing Bureau History

After merging the aggregated bureau features, 44,020 applicants had no corresponding records in `bureau.csv`. These missing values represent the absence of historical bureau records rather than unknown numerical values.

Therefore, the missing aggregated bureau features were replaced with `0`. This indicates that no historical bureau credit information was available for those applicants.

In [10]:
previous = pd.read_csv("../data/previous_application.csv")

print(previous.shape)
previous.head()

(1670214, 37)


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
print(previous.info())
print("\nUnique applicants:", previous["SK_ID_CURR"].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  str    
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  str    
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  str    
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-null   float6

In [12]:
previous_agg = previous.groupby("SK_ID_CURR").agg(
    PREV_APPLICATION_COUNT=("SK_ID_PREV", "count"),
    PREV_APPROVED_COUNT=("NAME_CONTRACT_STATUS", lambda x: (x == "Approved").sum()),
    PREV_REFUSED_COUNT=("NAME_CONTRACT_STATUS", lambda x: (x == "Refused").sum()),
    PREV_CANCELED_COUNT=("NAME_CONTRACT_STATUS", lambda x: (x == "Canceled").sum()),
    PREV_CREDIT_MEAN=("AMT_CREDIT", "mean"),
    PREV_CREDIT_MAX=("AMT_CREDIT", "max"),
    PREV_APPLICATION_MEAN=("AMT_APPLICATION", "mean"),
    PREV_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    PREV_DOWN_PAYMENT_MEAN=("AMT_DOWN_PAYMENT", "mean")
).reset_index()

In [13]:
enhanced_df = enhanced_df.merge(
    previous_agg,
    on="SK_ID_CURR",
    how="left"
)

print(enhanced_df.shape)

(307511, 140)


In [14]:
previous_features = [
    "PREV_APPLICATION_COUNT",
    "PREV_APPROVED_COUNT",
    "PREV_REFUSED_COUNT",
    "PREV_CANCELED_COUNT",
    "PREV_CREDIT_MEAN",
    "PREV_CREDIT_MAX",
    "PREV_APPLICATION_MEAN",
    "PREV_ANNUITY_MEAN",
    "PREV_DOWN_PAYMENT_MEAN"
]

enhanced_df[previous_features].isnull().sum()

PREV_APPLICATION_COUNT    16454
PREV_APPROVED_COUNT       16454
PREV_REFUSED_COUNT        16454
PREV_CANCELED_COUNT       16454
PREV_CREDIT_MEAN          16454
PREV_CREDIT_MAX           16454
PREV_APPLICATION_MEAN     16454
PREV_ANNUITY_MEAN         16871
PREV_DOWN_PAYMENT_MEAN    33906
dtype: int64

In [15]:
enhanced_df[previous_features] = enhanced_df[previous_features].fillna(0)

In [16]:
enhanced_df[previous_features].isnull().sum().sum()

np.int64(0)

In [17]:
enhanced_df.to_csv(
    "../data/application_enhanced.csv",
    index=False
)

print("Enhanced dataset saved!")

Enhanced dataset saved!


In [18]:
print(enhanced_df.shape)

(307511, 140)


## Enhanced Dataset

The aggregated historical credit information from `bureau.csv` and `previous_application.csv` was merged with the main application dataset.

The resulting dataset contains 307,511 applicants and 140 features. The additional features capture applicants' previous credit history, outstanding debt, overdue amounts, previous loan applications, and previous application outcomes.

This enhanced dataset will be used to evaluate whether historical credit behaviour improves credit-risk prediction compared with the application-only baseline.